## Libraries and Preparation

In [24]:
import numpy as np
import pandas as pd
import os
import kagglehub
from typing import List
import re, string, ast
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter

In [2]:
path = kagglehub.competition_download('kse-ua-location-extraction-2026')
print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/kse-ua-location-extraction-2026


In [3]:
uk_df = pd.read_csv('/kaggle/input/competitions/kse-ua-location-extraction-2026/uk_geo_dataset.csv')
print(uk_df.head(5))

                                                text loc_markers org_markers  \
0  Чим довше мають скачки тиску гіпертензією, тим...          []          []   
1  А поки що починали цвісти троянди, випускники ...          []          []   
2  Крім того, в 2020 р. багато експертів прогнозу...          []          []   
3  Сильно сумніваюся, що ви зупините свій вибір н...          []          []   
4  Цей унікальний правовий важіль утворено 1998 р...          []          []   

    per_markers  is_valid  
0            []         0  
1  [(114, 131)]         0  
2            []         0  
3            []         0  
4            []         0  


In [4]:
ru_df = pd.read_csv('/kaggle/input/competitions/kse-ua-location-extraction-2026/ru_geo_dataset.csv')
print(ru_df.head(5))

                                                text  \
0  Вице-премьер по социальным вопросам Татьяна Го...   
1  По словам Голиковой, чаще всего онкологические...   
2  Вице-премьер напомнила, что главные факторы см...   
3  В начале года стало известно, что смертность о...   
4  По данным Росстата, в 2017 году от рака умерли...   

                                         loc_markers   org_markers  \
0                                         [(82, 88)]  [(149, 160)]   
1  [(89, 98), (100, 108), (110, 118), (121, 139),...            []   
2                                         [(57, 63)]            []   
3                                                 []            []   
4                                                 []    [(10, 18)]   

  per_markers  doc_id  sent_id  
0  [(36, 52)]       0        0  
1  [(10, 19)]       0        1  
2          []       0        2  
3          []       0        3  
4          []       0        4  


In [5]:
labels_df = pd.read_csv('/kaggle/input/competitions/kse-ua-location-extraction-2026/labeling_sample.csv')
print(labels_df.head(5))

                                                text  \
0                                           -261.310   
1  Проспект Бандери стоїть, як і пів міста… довго...   
2  ☢️Радіаційний фон (мкР/год) по Київщині станом...   
3  Друзі. Останні дві ночі, і ще декілька наступн...   
4  Сьогодні з Києва до Хелма вперше вирушив новий...   

                                           locations  \
0                                                 []   
1                               ['Проспект Бандери']   
2  ['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', '...   
3                              ['проспекті Івасюка']   
4                                 ['Києва', 'Хелма']   

                                         loc_markers  
0                                                 []  
1                                          [(0, 16)]  
2  [(31, 39), (68, 77), (83, 87), (93, 100), (105...  
3                                       [(134, 151)]  
4                               [(11, 16), (20, 25)

In [6]:
test_df = pd.read_csv('/kaggle/input/competitions/kse-ua-location-extraction-2026/test.csv')
print(test_df.head(5))

   text_id                                               text locations
0        0  ❗️Кількість поранених зросла до трьох, – Кличк...        []
1        1  🥤В Києві за 91,13 млн гривень починаються робо...        []
2        2  ▪️Сьогодні вночі росіяни завдали ракетного уда...        []
3        3  Наразі у запасах росіян найбільше балістичних ...        []
4        4  ⛸В один день, 29 серпня, ДП "Центральна учбово...        []


# Exploratory Data Analysis

In [7]:
def to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

MARK_COLS = ['loc_markers', 'org_markers', 'per_markers']
for df in (uk_df, ru_df):
    for c in MARK_COLS:
        df[c] = df[c].apply(to_list)
labels_df['loc_markers'] = labels_df['loc_markers'].apply(to_list)
labels_df['locations'] = labels_df['locations'].apply(to_list)

In [8]:
uk_raw = uk_df.copy()
ru_raw = ru_df.copy()

## Do the same texts have same labels

In [9]:
for name, df in [('uk', uk_raw), ('ru', ru_raw)]:
    ann = df[MARK_COLS].astype(str).agg('|'.join, axis=1)
    print(f'conflicting texts ({name}):', (ann.groupby(df['text']).nunique() > 1).sum())

conflicting texts (uk): 0
conflicting texts (ru): 15002


In [10]:
ann = ru_raw[MARK_COLS].astype(str).agg('|'.join, axis=1)
conf = ann.groupby(ru_raw['text']).nunique()
conf_texts = conf[conf > 1].index[:5]

for t in conf_texts:
    print(ru_raw.loc[ru_raw['text'] == t, ['doc_id', 'sent_id'] + MARK_COLS].to_string())
    print('---')

         doc_id  sent_id loc_markers                       org_markers   per_markers
5052090  426903        7          []  [(1, 6), (125, 138), (216, 233)]  [(141, 156)]
5052102  426903       19          []          [(125, 138), (216, 233)]  [(141, 156)]
5052111  426903       28          []  [(1, 6), (125, 138), (216, 233)]  [(141, 156)]
---
         doc_id  sent_id loc_markers org_markers   per_markers
5357287  453496       10          []   [(1, 11)]  [(159, 166)]
6140766  535025        6          []          []  [(159, 166)]
---
         doc_id  sent_id   loc_markers          org_markers               per_markers
5170045  437002        9  [(249, 257)]  [(1, 17), (93, 96)]              [(157, 173)]
5170058  437002       22            []  [(1, 17), (93, 96)]  [(157, 173), (249, 257)]
5170070  437002       34  [(249, 257)]            [(1, 17)]              [(157, 173)]
5170077  437002       41  [(249, 257)]            [(1, 17)]              [(157, 173)]
5170084  437002       48         

## Data leakage in uk_df (is_valid)

In [11]:
leak_uk = set(uk_raw.loc[uk_raw.is_valid == 0, 'text']) & set(uk_raw.loc[uk_raw.is_valid == 1, 'text'])
print('uk leakage:', len(leak_uk))
uk_raw = uk_raw[~((uk_raw.is_valid == 1) & uk_raw['text'].isin(leak_uk))]

uk leakage: 52


## Duplicates

In [12]:
print(f"Overall count (uk): {uk_df['text'].count()}")
print(f"Distinct count (uk): {uk_df['text'].nunique()}")
print(f"Overall count (ru): {ru_df['text'].count()}")
print(f"Distinct count (ru): {ru_df['text'].nunique()}")
print(f"Overall count (label): {labels_df['text'].count()}")
print(f"Distinct count (label): {labels_df['text'].nunique()}")

Overall count (uk): 1010000
Distinct count (uk): 1007546
Overall count (ru): 8028840
Distinct count (ru): 7505339
Overall count (label): 26
Distinct count (label): 26


In [13]:
uk_final = uk_raw.drop_duplicates(subset=['text'])
print(f"Overall count (uk): {uk_final['text'].count()}")
print(f"Distinct count (uk): {uk_final['text'].nunique()}")

Overall count (uk): 1007546
Distinct count (uk): 1007546


In [14]:
ru_final = ru_raw.drop_duplicates(subset=['text'])
print(f"Overall count (ru): {ru_final['text'].count()}")
print(f"Distinct count (ru): {ru_final['text'].nunique()}")

Overall count (ru): 7505339
Distinct count (ru): 7505339


In [15]:
uk_tr = uk_final[uk_final.is_valid == 0]
uk_va = uk_final[uk_final.is_valid == 1]

## data leakage in ru

In [16]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
tr_idx, va_idx = next(gss.split(ru_raw, groups=ru_raw['doc_id']))
ru_tr_raw = ru_raw.iloc[tr_idx]
ru_va_raw = ru_raw.iloc[va_idx]

leak_ru = set(ru_tr_raw['text']) & set(ru_va_raw['text'])
print('ru leakage:', len(leak_ru))
ru_va_raw = ru_va_raw[~ru_va_raw['text'].isin(leak_ru)]

ru_tr = ru_tr_raw.drop_duplicates(subset=['text'])
ru_va = ru_va_raw.drop_duplicates(subset=['text'])

ru leakage: 56141


## Competition metrics

In [17]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p,  = 0.0, 0.0, 0.0, 0.0

    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = set([hard_processing(s) for s in y_true_sample])
        y_pred_sample = set([hard_processing(s) for s in y_pred_sample])

        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)

    if tp + fp == 0:
        if p == 0:
            precision = 1.0
        else:
            precision = 0.0
    else:
        precision = tp/(tp + fp)

    if tp + fn == 0:
        if p == 0:
            recall = 1.0
        else:
            recall = 0.0
    else:
        recall = tp/(tp + fn)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 8)

def make_gt(df):
    return [[text[l:r] for l, r in spans]
            for text, spans in zip(df['text'], df['loc_markers'])]

## entities and empty texts

In [18]:
splits = {
    'uk': uk_df, 'ru': ru_df,
    'uk train': uk_tr, 'uk valid': uk_va,
    'ru train': ru_tr, 'ru valid': ru_va,
}

for name, df in splits.items():
    n_spans = df['loc_markers'].apply(len)
    print(f'rows ({name}):', len(df))
    print(f'without loc_markers ({name}):', f"{(n_spans == 0).mean():.2%}")
    print(f'loc spans ({name}):', n_spans.sum())
    print(f'per text ({name}):', round(n_spans.mean(), 2))

rows (uk): 1010000
without loc_markers (uk): 76.89%
loc spans (uk): 345114
per text (uk): 0.34
rows (ru): 8028840
without loc_markers (ru): 69.31%
loc spans (ru): 3708173
per text (ru): 0.46
rows (uk train): 997598
without loc_markers (uk train): 76.87%
loc spans (uk train): 341285
per text (uk train): 0.34
rows (uk valid): 9948
without loc_markers (uk valid): 76.91%
loc spans (uk valid): 3293
per text (uk valid): 0.33
rows (ru train): 6778181
without loc_markers (ru train): 69.12%
loc spans (ru train): 3134513
per text (ru train): 0.46
rows (ru valid): 727158
without loc_markers (ru valid): 69.01%
loc spans (ru valid): 336199
per text (ru valid): 0.46


## cuts check

In [19]:
bad_left = bad_right = 0
sample = uk_df.sample(50_000, random_state=42)

for text, spans in zip(sample['text'], sample['loc_markers']):
    for l, r in spans:
        if l > 0 and text[l - 1].isalpha():
            bad_left += 1
        if r < len(text) and text[r].isalpha():
            bad_right += 1
        
print('span starts inside word:', bad_left)
print('span ends inside word:', bad_right)

span starts inside word: 103
span ends inside word: 174


## language and scripts

In [20]:
patterns = {
    'uk letters іїєґ': '[іїєґІЇЄҐ]',
    'ru letters ыъэё': '[ыъэёЫЪЭЁ]',
    'latin': '[A-Za-z]',
}

for name, df in [('uk', uk_df), ('ru', ru_df)]:
    s = df['text'].sample(200_000, random_state=42)
    for k, p in patterns.items():
        print(f'{name} - {k}: {s.str.contains(p, regex=True).mean():.2%}')

uk - uk letters іїєґ: 98.74%
uk - ru letters ыъэё: 0.09%
uk - latin: 6.63%
ru - uk letters іїєґ: 0.01%
ru - ru letters ыъэё: 81.59%
ru - latin: 16.98%


## text lengths

In [21]:
  for name, df in [('uk', uk_raw), ('ru', ru_raw)]:
      L = df['text'].str.len()
      print(f'length quantiles ({name}):', L.quantile([.5, .9, .99, .999]).round(0).to_dict())

length quantiles (uk): {0.5: 91.0, 0.9: 170.0, 0.99: 238.0, 0.999: 253.0}
length quantiles (ru): {0.5: 110.0, 0.9: 195.0, 0.99: 300.0, 0.999: 444.0}


## what the metric does to real location names?

In [25]:
gt = [s for row in make_gt(sample) for s in row]
cleaned = [hard_processing(s) for s in gt]

print('strings:', len(gt))
print('empty after normalization:', cleaned.count(''))
print('top-20 normalized strings:', Counter(cleaned).most_common(20))

strings: 16972
empty after normalization: 19
top-20 normalized strings: [('україни', 1207), ('україні', 850), ('сша', 442), ('росії', 383), ('києві', 361), ('україна', 349), ('україну', 202), ('криму', 197), ('києва', 196), ('рф', 166), ('росія', 151), ('донбасі', 151), ('польщі', 136), ('київ', 126), ('європи', 112), ('росією', 110), ('франції', 105), ('львові', 97), ('львова', 93), ('укрїн', 88)]


## labeling_sample consistency

In [29]:
lab_gt = make_gt(labels_df)
same = [{hard_processing(s) for s in a} == {hard_processing(s) for s in b}
        for a, b in zip(lab_gt, labels_df['locations'])]
print('spans vs locations agreement:', np.mean(same))

spans vs locations agreement: 1.0


## overlaps

In [30]:
uk_txt = set(uk_df['text'])
ru_txt = set(ru_df['text'])
train_txt = uk_txt | ru_txt

print('overlap with test:', len(train_txt & set(test_df['text'])))
print('overlap with labeling_sample:', len(train_txt & set(labels_df['text'])))
print('overlap uk vs ru:', len(uk_txt & ru_txt))

overlap with test: 0
overlap with labeling_sample: 0
overlap uk vs ru: 0


## bootstrap on labeling_sample

In [31]:
def bootstrap(y_true, y_pred, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        scores.append(f1_score_comp([y_true[i] for i in idx], [y_pred[i] for i in idx]))
    return np.percentile(scores, [2.5, 97.5])

## Metric Justification¶

#### Metric Justification

`comp_metric` pools true positives, false positives and false negatives across the whole dataset before computing F1, so frequent places carry more weight. Its main weaknesses are the exact-string match, which gives no partial credit and treats case forms as different entities, and the normalization collisions described above.

For complementary metrics, span-level precision and recall with exact offsets would show whether errors come from wrong boundaries or from missed and invented entities, and per-language F1 would show how the two languages differ. As an alternative development metric, I propose span-level micro F1 matched on (text id, left, right). It avoids the normalization collisions, and we would still report `comp_metric` so the numbers stay comparable with the leaderboard.

## Validation

#### Validation

The Russian validation is a 10% holdout grouped by `doc_id`, with leaked texts removed, giving 727,158 rows. The Ukrainian validation is the provided `is_valid` split, minus the 52 leaked texts, giving 9,948 rows and 3,293 gold spans. Because that holdout is small, differences need to be checked with the bootstrap before we trust them.

We expect a high correlation with the leaderboard, since the metric is the same and there is no text overlap between training and validation. We expect a lower correlation for three reasons: the Ukrainian holdout is small, micro F1 is dominated by frequent case forms, and label noise affects validation and the leaderboard differently. Adversarial validation has not been run yet. If it is run, it would use the same model as the main task with a binary head that separates train rows from test rows, and an AUC near 0.5 would mean the split is sound.